# 12 - Synthetic Benchmark Validation Framework Summary

This notebook is intentionally table-driven. It reads the artifacts produced by
`scripts/validate_synthetic_benchmark.py` and `scripts/build_benchmark_registries.py`
and presents release readiness, warnings, and discrepancy details without
reimplementing validation logic in notebook cells.

Read the statuses with the specification's decision rule in mind:

- **PASS** — the interval lies inside the predeclared tolerance, or the invariant
  holds exactly. For metrics carrying no bootstrap interval this means "not
  obviously discrepant", which is weaker than equivalence.
- **WARNING** — the estimate crosses a tolerance, the evidence is too thin to
  decide, or a non-critical discrepancy remains.
- **FAIL** — a critical invariant broke, truth leaked, a real record was copied,
  or a core benchmark property is materially outside tolerance.
- **INCONCLUSIVE** — the check could not be run on the available data. Never read
  it as a pass.

Do not average these into one score: a high average hides a fatal defect.


In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VERSION = "v0_3_temporal_candidate_revision"
BASE = PROJECT_ROOT / "reports/tables/synthetic_benchmark" / VERSION
VALIDATION_DIR = BASE / "validation_framework"
REGISTRY_DIR = BASE / "registries"

metrics = pd.read_csv(VALIDATION_DIR / "validation_metrics_long.csv")
gates = pd.read_csv(VALIDATION_DIR / "validation_gate_summary.csv")
discrepancies = pd.read_csv(VALIDATION_DIR / "discrepancy_register.csv")
probes = pd.read_csv(VALIDATION_DIR / "probe_linker_results.csv")
manifest = json.loads((VALIDATION_DIR / "validation_manifest.json").read_text())

{k: v for k, v in manifest.items() if k not in {"input_checksums", "probe_linker_results"}}

{'benchmark_version': 'v0_3_temporal_candidate_revision',
 'scenario': 'central_provisional',
 'world': '001',
 'corruption': '001',
 'synthetic_dir': 'data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisional/world_001/corruption_001',
 'generated_at_utc': '2026-07-24T08:21:01.787066+00:00',
 'strict_60m': False,
 'bootstrap_reps': 200,
 'robustness_sweep': True,
 'canonical_replay_checked': True,
 'metric_count': 161,
 'gate_count': 15,
 'overall_status': 'PASS_WITH_WARNINGS',
 'n_metric_failures': 18,
 'n_metric_failures_in_noncritical_gates': 18,
 'noncritical_gates_with_metric_failures': ['marginals',
  'missingness_text_identifier',
  'buyer_activity',
  'text'],
 'notes': ['Covers the Section 7 minimum suite: internal integrity, marginals, conditionals, missingness, temporal, buyer activity, candidate environment, text, names/identifiers, hidden-truth difficulty, algorithm utility and robustness.',
  'Still out of scope and therefore never reported 

## Release gate

Critical gates block release on their own. Non-critical gates can only downgrade
the run to `PASS_WITH_WARNINGS`, and their breaches belong in the fidelity budget:
the benchmark is allowed to simplify the real corpus as long as the simplification
is written down.

In [2]:
gates.sort_values(["critical", "status", "gate"], ascending=[False, True, True])

,gate,status,critical,n_pass,n_warning,n_fail,n_inconclusive,headline,blocking_reason,warning_reason
13,algorithm_utility,PASS,True,15,0,0,0,algorithm_utility validation PASS,NaN,NaN
5,candidate_environment,PASS,True,10,0,0,0,candidate_environment validation PASS,NaN,NaN
3,conditionals,PASS,True,4,0,0,0,conditionals validation PASS,NaN,NaN
0,internal,PASS,True,34,0,0,0,internal validation PASS,NaN,NaN
11,privacy,PASS,True,4,0,0,0,privacy validation PASS,NaN,NaN
1,specification_recovery,PASS,True,6,0,0,0,specification_recovery validation PASS,NaN,NaN
12,hidden_truth_difficulty,WARNING,True,9,1,0,0,hidden_truth_difficulty validation WARNING,NaN,blocking_pairs_completeness:PC
4,temporal,WARNING,True,7,1,0,0,temporal validation WARNING,NaN,followup_runway:abs_diff_pp
8,missingness_structure,PASS,False,4,0,0,0,missingness_structure validation PASS,NaN,NaN
7,buyer_activity,WARNING,False,7,1,4,0,buyer_activity validation WARNING,NaN,activity_gini:abs_diff; activity_share:abs_diff_pp; activity_share:abs_diff_pp; relati...


### What is currently blocking

In [3]:
blocking = gates[gates["critical"] & gates["status"].eq("FAIL")]
if blocking.empty:
    print("No critical gate is failing.")
else:
    display(blocking[["gate", "n_fail", "blocking_reason"]])
    display(
        metrics[metrics["scope"].isin(blocking["gate"]) & metrics["status"].eq("FAIL")][
            ["property", "metric", "synthetic_estimate", "ci_low", "ci_high", "tolerance", "notes"]
        ]
    )

No critical gate is failing.


### Metric failures carried under non-critical gates

A non-critical gate reports `WARNING` even when individual metrics fail, so the
headline status alone understates how many metric-level failures the release is
carrying. These do not block under the documented gate policy, but they are the
fidelity debt the benchmark is shipping with.

In [4]:
print(
    manifest["n_metric_failures"],
    "metric failures total;",
    manifest["n_metric_failures_in_noncritical_gates"],
    "inside non-critical gates:",
    manifest["noncritical_gates_with_metric_failures"],
)
carried = gates[(~gates["critical"]) & (gates["n_fail"] > 0)]
display(carried[["gate", "status", "n_pass", "n_fail", "warning_reason"]])

18 metric failures total; 18 inside non-critical gates: ['marginals', 'missingness_text_identifier', 'buyer_activity', 'text']


,gate,status,n_pass,n_fail,warning_reason
2,marginals,WARNING,8,9,department:TV; department:JS; duration:W1_scaled; duration:q50_relative_error; duratio...
6,missingness_text_identifier,WARNING,3,3,siren_missing:rate_abs_diff_pp; text_length:q50_relative_error; siren_present:presence...
7,buyer_activity,WARNING,7,4,activity_gini:abs_diff; activity_share:abs_diff_pp; activity_share:abs_diff_pp; relati...
10,text,WARNING,5,2,bigram_distribution:JS; internal_duplicate_share:abs_diff_pp; text_length_chars:q50_re...


### Reproducibility

`canonical_replay_matches_released_tables` regenerates the benchmark from the
recorded seeds and the live scenario file and compares canonical table content.
`scenario_snapshot_matches_scenario_file` checks that the scenario snapshot
saved with the run still matches that file, so configuration drift is named
rather than surfacing as an unexplained hash mismatch.

In [5]:
print("canonical replay checked:", manifest["canonical_replay_checked"])
metrics[metrics["property"].eq("reproducibility")][["metric", "status", "notes"]]

canonical replay checked: True


,metric,status,notes
30,manifest_records_seeds_and_code_version,PASS,missing=[]
31,scenario_snapshot_matches_scenario_file,PASS,recorded scenario snapshot matches the live scenario file
32,corruption_log_seed_matches_manifest,PASS,declared=20260722; logged=[20260722]
33,canonical_replay_matches_released_tables,PASS,9/9 tables reproduced from the recorded seeds and the live scenario file


## Discrepancy register

In [6]:
discrepancies.sort_values(["status", "scope", "property", "metric"])

,scope,subgroup,property,metric,real_estimate,synthetic_estimate,difference,effect_size,tolerance,status,provenance,notes
18,buyer_activity,overall,activity_gini,abs_diff,0.831614,0.563260,-0.268354,0.268354,0.1,FAIL,observable_real_vs_synthetic,bootstrap_reps=200; n_real_buyers=5268; n_syn_buyers=2663
20,buyer_activity,top5pct,activity_share,abs_diff_pp,0.661286,0.366487,-29.479877,29.479877,10.0,FAIL,observable_real_vs_synthetic,share of all notices held by the top5pct most active buyers
21,buyer_activity,top10pct,activity_share,abs_diff_pp,0.779823,0.484109,-29.571407,29.571407,10.0,FAIL,observable_real_vs_synthetic,share of all notices held by the top10pct most active buyers
22,buyer_activity,overall,relative_notices_per_buyer,q99_abs_diff,16.205591,6.011502,-10.194089,10.194089,1.0,FAIL,observable_real_vs_synthetic,activity divided by that corpus's own mean activity (scale-free)
1,marginals,overall,department,JS,NaN,NaN,0.155895,0.155895,0.05,FAIL,observable_real_vs_synthetic,NaN
0,marginals,overall,department,TV,NaN,NaN,0.206540,0.206540,0.05,FAIL,observable_real_vs_synthetic,NaN
2,marginals,overall,duration,W1_scaled,NaN,NaN,1.006133,1.006133,0.1,FAIL,observable_real_vs_synthetic,NaN
3,marginals,overall,duration,q50_relative_error,6.000000,12.000000,6.000000,1.000000,0.1,FAIL,observable_real_vs_synthetic,NaN
4,marginals,overall,duration,q75_relative_error,12.000000,18.920457,6.920457,0.576705,0.1,FAIL,observable_real_vs_synthetic,NaN
5,marginals,overall,duration,q90_relative_error,48.000000,36.000000,-12.000000,0.250000,0.1,FAIL,observable_real_vs_synthetic,NaN


## Metric inventory by gate

In [7]:
(
    metrics.groupby(["scope", "status"])
    .size()
    .rename("n")
    .reset_index()
    .pivot(index="scope", columns="status", values="n")
    .fillna(0)
    .astype(int)
)

status,FAIL,INCONCLUSIVE,PASS,WARNING
scope,,,,
algorithm_utility,0,0,15,0
buyer_activity,4,0,7,1
candidate_environment,0,0,10,0
conditionals,0,0,4,0
hidden_truth_difficulty,0,0,9,1
internal,0,0,34,0
marginals,9,0,8,1
missingness_structure,0,0,4,0
missingness_text_identifier,3,0,3,4


## Benchmark difficulty

These need the sealed truth tables and cannot be computed on real BOAMP at all.
Blocking recall says whether a comparison model is even being given the chance to
work; score overlap says whether telling a match from a plausible non-match takes
real discrimination.

In [8]:
metrics[metrics["scope"].eq("hidden_truth_difficulty")][
    ["subgroup", "property", "metric", "synthetic_estimate", "tolerance", "status"]
]

,subgroup,property,metric,synthetic_estimate,tolerance,status
130,algorithm_scope,true_matches_in_scope,count,152,context_only,PASS
131,production_window,blocking_pairs_completeness,PC,0.3618421052631579,0.25,PASS
132,production_window,pairs_quality,PQ,0.21825396825396826,0.05,PASS
133,production_window,reduction_ratio,RR,0.9967615498297243,0.95,PASS
134,36m_window,blocking_pairs_completeness,PC,0.5789473684210527,0.8,WARNING
135,36m_window,pairs_quality,PQ,0.1317365269461078,0.05,PASS
136,36m_window,reduction_ratio,RR,0.9914155368502217,0.95,PASS
137,production_window,match_vs_hard_negative_score,overlap,0.2084910013844024,0.05-0.95,PASS
138,production_window,true_match_rank,MRR,0.9454545454545454,context_only,PASS
139,production_window,top1_top2_margin,median,0.34576760661756034,context_only,PASS


## Probe linkers

Three deliberately different linkers with frozen parameters. None of them is the
production acceptance threshold — a pipeline threshold is an output of that
pipeline, never a definition of truth. The benchmark is informative if the probes
separate, and unusable if any of them solves it.

In [9]:
probes

,probe,n_predicted_pairs,pair_precision,pair_recall,pair_f1,bcubed_precision,bcubed_recall,bcubed_f1
0,deterministic_top1,71,0.647887,0.302632,0.412556,0.943966,0.725316,0.820322
1,score_threshold_all_ranks,56,0.732143,0.269737,0.394231,0.971730,0.713080,0.822551
2,fellegi_sunter,94,0.478723,0.296053,0.365854,0.884473,0.729958,0.799821


## Predeclared tolerances and parameter provenance

Tolerances are read from the gate code itself, so a retuned tolerance shows up here
instead of quietly diverging from the registry. Parameters marked
`SCENARIO_UNIDENTIFIED` encode assumptions real BOAMP cannot settle and must be
varied across scenarios rather than quoted as estimates.

In [10]:
tolerances = pd.read_csv(REGISTRY_DIR / "tolerance_registry.csv")
parameters = pd.read_csv(REGISTRY_DIR / "parameter_registry.csv")

display(tolerances[tolerances["gate_is_critical"]][["module", "tolerance_key", "value", "gate"]].drop_duplicates())
parameters["provenance_class"].value_counts()

,module,tolerance_key,value,gate
1,fidelity,categorical_tv,0.05,conditionals
2,fidelity,categorical_tv,0.05,temporal
3,fidelity,categorical_tv,0.05,candidate_environment
6,fidelity,categorical_js,0.05,conditionals
7,fidelity,categorical_js,0.05,temporal
...,...,...,...,...
189,difficulty,probe_best_f1_max,0.99,algorithm_utility
190,difficulty,probe_best_f1_min,0.05,hidden_truth_difficulty
191,difficulty,probe_best_f1_min,0.05,algorithm_utility
192,difficulty,probe_f1_spread_min,0.01,hidden_truth_difficulty


provenance_class
EMPIRICAL_OBSERVABLE                   43
SCENARIO_UNIDENTIFIED                  18
SCENARIO_TUNED_TO_CANDIDATE_TARGETS    15
DERIVED_RESWEPT                         6
UNCLASSIFIED                            1
Name: count, dtype: int64

### Parameters whose scenario-file value was overridden at run time

In [11]:
parameters[parameters["overridden_at_runtime"]][
    ["parameter", "scenario_file_value", "effective_value", "provenance_class"]
]

,parameter,scenario_file_value,effective_value,provenance_class
7,recurrence.scoped_candidate_environment.cpv_divisions,"['32', '35', '48', '72']","['32', '35', '48', '72']",SCENARIO_TUNED_TO_CANDIDATE_TARGETS
8,recurrence.scoped_candidate_environment.recurrence_propensity_multiplier,1.0,1.3,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
9,recurrence.scoped_candidate_environment.near_window_share,0.0,0.75,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
15,recurrence.scoped_candidate_environment.hard_negative_alignment_rate,0.0,0.1,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
16,recurrence.scoped_candidate_environment.cluster_scoped_needs_by_buyer,False,True,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
17,recurrence.scoped_candidate_environment.scoped_buyer_affinity_share,0.07,0.07,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
18,recurrence.scoped_candidate_environment.scoped_need_probability_high,0.7,0.7,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
19,recurrence.scoped_candidate_environment.stabilize_scoped_establishment,False,True,SCENARIO_TUNED_TO_CANDIDATE_TARGETS
20,recurrence.scoped_candidate_environment.stabilize_scoped_name_fallback,False,True,SCENARIO_TUNED_TO_CANDIDATE_TARGETS


## Replicate inventory

Seed-to-seed stability cannot be estimated from a single world per scenario, so it
is reported `INCONCLUSIVE` rather than assumed.

In [12]:
display(pd.read_csv(REGISTRY_DIR / "scenario_manifest.csv"))
metrics[metrics["scope"].eq("robustness")][["subgroup", "property", "metric", "synthetic_estimate", "status", "notes"]]

,benchmark_version,scenario,world,corruption,world_seed,corruption_seed,generator_version,git_commit,n_observed_notices,path
0,v0_3_temporal_candidate_revision,central_provisional,1,1,20260721,20260722,0.3.0-temporal-candidate-revision,b3afe7c19a2efb44f0e1e37b874b7ddc3f891781,9471,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/central_provisiona...
1,v0_3_temporal_candidate_revision,clean_sanity,1,1,20260721,20260722,0.3.0-temporal-candidate-revision,b3afe7c19a2efb44f0e1e37b874b7ddc3f891781,2568,data/processed/synthetic_benchmark/v0_3_temporal_candidate_revision/clean_sanity/world...


,subgroup,property,metric,synthetic_estimate,status,notes
155,all_replicates,replicate_inventory,count,2,PASS,"scenarios=['central_provisional', 'clean_sanity']; seed replicates per scenario={'cent..."
156,seed_replicates,seed_stability,availability,1,INCONCLUSIVE,"no scenario has two independent world/corruption draws, so seed-to-seed stability cann..."
157,cross_scenario,blocking_pairs_completeness,coefficient_of_variation,1.3123848264750797,WARNING,"n_replicates=2; values=central_provisional/001-001=0.362, clean_sanity/001-001=0.014. ..."
158,cross_scenario,match_vs_hard_negative_score,coefficient_of_variation,NaN,INCONCLUSIVE,only 1 evaluable replicate(s)
159,cross_scenario,probe_headroom,coefficient_of_variation,1.2424903800103395,WARNING,"n_replicates=2; values=central_provisional/001-001=0.413, clean_sanity/001-001=0.027. ..."
160,ranking_stability,probe_ranking,min_kendall_tau,NaN,INCONCLUSIVE,central_provisional vs clean_sanity: tau=nan. Rank agreement across replicates that di...
